In [3]:
import sys
import os

# Check for a modern PEP 405 virtual environment
is_venv = sys.prefix != sys.base_prefix

print(f"Is inside a venv? -> {is_venv}")
print(f"Current Environment Path: {sys.prefix}")
print(f"Active Python Executable: {sys.executable}")

# Optional check: If you activated it via terminal scripts
if 'VIRTUAL_ENV' in os.environ:
    print(f"Activated via terminal VIRTUAL_ENV: {os.environ['VIRTUAL_ENV']}")


Is inside a venv? -> True
Current Environment Path: c:\Users\arz_2\Basira_Motassim_Group_A\.venv
Active Python Executable: c:\Users\arz_2\Basira_Motassim_Group_A\.venv\Scripts\python.exe
Activated via terminal VIRTUAL_ENV: C:\Users\arz_2\Basira_Motassim_Group_A\.venv


In [6]:
import pandas as pd, numpy as np
from scipy import stats

CLINICAL_FILE = r"C:\Users\arz_2\Basira_Motassim_Group_A\data_rna\clinical.cart.2026-10-06\clinical.tsv"   # edit this
clin = pd.read_csv(CLINICAL_FILE, sep="\t", comment="#")
print(clin.shape)
print(list(clin.columns))
clin.head()

(1503, 202)
['project.project_id', 'cases.case_id', 'cases.consent_type', 'cases.days_to_consent', 'cases.days_to_lost_to_followup', 'cases.disease_type', 'cases.index_date', 'cases.lost_to_followup', 'cases.primary_site', 'cases.submitter_id', 'demographic.age_at_index', 'demographic.age_is_obfuscated', 'demographic.cause_of_death', 'demographic.cause_of_death_source', 'demographic.country_of_birth', 'demographic.country_of_residence_at_enrollment', 'demographic.days_to_birth', 'demographic.days_to_death', 'demographic.demographic_id', 'demographic.education_level', 'demographic.ethnicity', 'demographic.marital_status', 'demographic.occupation_duration_years', 'demographic.population_group', 'demographic.race', 'demographic.sex_at_birth', 'demographic.submitter_id', 'demographic.vital_status', 'demographic.year_of_birth', 'demographic.year_of_birth_range', 'demographic.year_of_death', 'diagnoses.adrenal_hormone', 'diagnoses.age_at_diagnosis', 'diagnoses.ajcc_clinical_m', 'diagnoses.aj

,project.project_id,cases.case_id,cases.consent_type,cases.days_to_consent,cases.days_to_lost_to_followup,cases.disease_type,cases.index_date,cases.lost_to_followup,cases.primary_site,cases.submitter_id,...,treatments.treatment_effect,treatments.treatment_effect_indicator,treatments.treatment_frequency,treatments.treatment_id,treatments.treatment_intent_type,treatments.treatment_or_therapy,treatments.treatment_outcome,treatments.treatment_outcome_duration,treatments.treatment_type,treatments.treatment_type_administered
0,TCGA-STAD,00781a96-4068-427c-a9c5-584d167c3dea,Informed Consent,38,'--,Adenomas and Adenocarcinomas,Diagnosis,Yes,Stomach,TCGA-BR-8368,...,'--,'--,'--,0943354a-72ff-5607-a89d-69a865781d4f,Adjuvant,unknown,'--,'--,"Pharmaceutical Therapy, NOS",'--
1,TCGA-STAD,00781a96-4068-427c-a9c5-584d167c3dea,Informed Consent,38,'--,Adenomas and Adenocarcinomas,Diagnosis,Yes,Stomach,TCGA-BR-8368,...,'--,'--,'--,2c0c1227-0722-42a5-803e-a3e1a85255e6,Adjuvant,unknown,'--,'--,"Radiation Therapy, NOS",'--
2,TCGA-STAD,00ee3098-1b32-4e7a-81ea-993773587c41,Informed Consent,0,'--,Adenomas and Adenocarcinomas,Diagnosis,No,Stomach,TCGA-D7-8575,...,'--,'--,'--,b1bfba16-7f97-48b6-9463-bc10e6641bab,'--,no,'--,'--,"Radiation Therapy, NOS",'--
3,TCGA-STAD,00ee3098-1b32-4e7a-81ea-993773587c41,Informed Consent,0,'--,Adenomas and Adenocarcinomas,Diagnosis,No,Stomach,TCGA-D7-8575,...,'--,'--,'--,cf5f0d7c-08e9-4fc6-a673-8d48638c7419,'--,no,'--,'--,"Pharmaceutical Therapy, NOS",'--
4,TCGA-STAD,00ee3098-1b32-4e7a-81ea-993773587c41,Informed Consent,0,'--,Adenomas and Adenocarcinomas,Diagnosis,No,Stomach,TCGA-D7-8575,...,'--,'--,'--,daf093e0-cb44-43b6-9967-edd28f64d66a,'--,no,'--,'--,"Surgery, NOS",'--


In [7]:
# GDC marks missing values as "'--"
clin = clin.replace("'--", np.nan)

print(clin["demographic.sex_at_birth"].value_counts(dropna=False), "\n")
print(clin["diagnoses.ajcc_pathologic_stage"].value_counts(dropna=False), "\n")
print(clin["diagnoses.primary_diagnosis"].value_counts(dropna=False))

demographic.sex_at_birth
male      1016
female     487
Name: count, dtype: int64 

diagnoses.ajcc_pathologic_stage
NaN           465
Stage IIIA    202
Stage IIIB    156
Stage IIB     154
Stage IIA     110
Stage IV       99
Stage IIIC     94
Stage IB       93
Stage II       77
Stage IA       31
Stage I        16
Stage III       6
Name: count, dtype: int64 

diagnoses.primary_diagnosis
Adenocarcinoma, NOS                   541
Carcinoma, diffuse type               263
Tubular adenocarcinoma                255
Adenocarcinoma, intestinal type       241
Mucinous adenocarcinoma                70
Signet ring cell carcinoma             47
Not Reported                           42
Papillary adenocarcinoma, NOS          21
Infiltrating duct carcinoma, NOS        6
Basal cell carcinoma, NOS               6
Papillary renal cell carcinoma          6
Seminoma, NOS                           3
Adenocarcinoma with mixed subtypes      2
Name: count, dtype: int64


In [8]:
def stage_group(s):
    if pd.isna(s): return np.nan
    s = str(s).upper().replace("STAGE", "").strip()
    for k in ("IV", "III", "II", "I"):      # order matters
        if s.startswith(k): return k
    return np.nan

clin["stage_grp"] = clin["diagnoses.ajcc_pathologic_stage"].map(stage_group)

COL = {
    "id":    "cases.submitter_id",
    "age":   "demographic.age_at_index",
    "sex":   "demographic.sex_at_birth",
    "stage": "stage_grp",
}

clin["pid"] = clin[COL["id"]].astype(str).str[:12]
clin = clin.drop_duplicates("pid").set_index("pid")
clin[COL["age"]] = pd.to_numeric(clin[COL["age"]], errors="coerce")

### Loading MAF files

In [13]:
import glob, pandas as pd

MAF_DIR = "mutation/maf_files"     # edit this
files = glob.glob("../mutation/maf_files/**/*.maf.gz", recursive=True)
print(len(files), "MAF files")     # you reported 434

maf = pd.concat(
    (pd.read_csv(f, sep="\t", comment="#", low_memory=False,
                 usecols=["Hugo_Symbol", "Variant_Classification", "Tumor_Sample_Barcode"])
     for f in files),
    ignore_index=True)
maf["patient"] = maf["Tumor_Sample_Barcode"].str[:12]
print(maf["patient"].nunique(), "patients with any MAF record")

434 MAF files
431 patients with any MAF record


In [ ]:
RNA_FILE = r"..\data_rna\gdc_sample_sheet.2026-10-11.tsv"   # edit this

cols = pd.read_csv(RNA_FILE, sep="\t", nrows=0).columns   # header only
rna_patients = {c[:12] for c in cols}

patients = set(maf["patient"]) & rna_patients
print(len(patients), "patients with mutation calls and RNA-seq")   # expect 404

0 patients with mutation calls and RNA-seq


In [25]:
SAMPLE_SHEET  = r"..\data_rna\gdc_sample_sheet.2026-10-11.tsv"   # edit this


ss = pd.read_csv(SAMPLE_SHEET, sep="\t")
print(ss["Tumor Descriptor"].value_counts())          # check what is in there

prim = ss[ss["Tumor Descriptor"] == "Primary"]
rna_patients = set(prim["Case ID"].str[:12])

patients = set(maf["patient"]) & rna_patients
print(len(patients), "patients with mutation calls and RNA-seq")   # expect about 404  # expect 404

Tumor Descriptor
Primary    412
Name: count, dtype: int64
401 patients with mutation calls and RNA-seq


In [23]:
ss.head()

,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
0,6a8033f9-8dcd-404a-ad51-b9063f92eff6,e4ccb76d-90cd-48d5-83d7-09d1aab4f439.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-STAD,TCGA-VQ-A8P8,TCGA-VQ-A8P8-01A,Tumor,Primary,Unknown,Unknown
1,fa2364e1-1d9c-43c3-8836-c4f6b204160d,8246417a-da0a-4330-af7e-cb382a28cc3f.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-STAD,TCGA-CD-5813,TCGA-CD-5813-01A,Tumor,Primary,Unknown,Unknown
2,0a51d6e0-82de-43b1-b72e-790d57ca9018,5c84144f-68b0-4f96-8b53-5a43a6f83847.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-STAD,TCGA-BR-6710,TCGA-BR-6710-01A,Tumor,Primary,Solid Tissue,Unknown
3,ff36547a-5887-46af-b0f5-99ea4c1baa96,b13b3f37-a06f-4484-81e3-c0d900d2f35c.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-STAD,TCGA-CG-4476,TCGA-CG-4476-01A,Tumor,Primary,Unknown,Unknown
4,2aa1c46b-2145-4009-be34-0e93dbfd462e,26080a1b-d146-4ffb-9276-dbc2343b717f.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-STAD,TCGA-CG-4460,TCGA-CG-4460-01A,Tumor,Primary,Unknown,Unknown


In [32]:
CAD_GENES = ["CDH1","CDH11","CDH13","CDH17","CDH2","CDH3","CDHR2","CDHR5",
             "CLSTN1","DSC2","DSG2","FAT1","PCDH1","PCDH17","PCDH18","PCDHGA10"]

NON_SILENT = ["Missense_Mutation","Nonsense_Mutation","Frame_Shift_Del","Frame_Shift_Ins",
              "Splice_Site","In_Frame_Del","In_Frame_Ins","Translation_Start_Site"]

ns = maf[maf["Variant_Classification"].isin(NON_SILENT) & maf["patient"].isin(patients)]

tp53 = set(ns.loc[ns["Hugo_Symbol"] == "TP53", "patient"])
cad  = set(ns.loc[ns["Hugo_Symbol"].isin(CAD_GENES), "patient"])

print(len(patients), len(tp53), len(cad))   # expect 404, 183, 164

401 183 164


In [33]:
# ---- Cell: Four groups ----
cad_only  = cad - tp53
tp53_only = tp53 - cad
co_mut    = tp53 & cad
wild_type = patients - tp53 - cad
n_patients = patients


In [34]:
def group_of(p):
    t, c = p in tp53, p in cad
    return "Co-mutated" if t and c else "TP53 only" if t else "Cadherin only" if c else "Wild-type"

grp = pd.Series({p: group_of(p) for p in patients}, name="group")
order = ["Co-mutated", "TP53 only", "Cadherin only", "Wild-type"]

df = clin.reindex(grp.index).join(grp)
print("Patients without a clinical record:", df[COL["age"]].isna().sum(), "(check this)")
print(df["group"].value_counts().reindex(order))

Patients without a clinical record: 5 (check this)
group
Co-mutated        76
TP53 only        107
Cadherin only     88
Wild-type        130
Name: count, dtype: int64


In [35]:
cont = [k for k in ("age", "purity") if k in COL]
cat  = [k for k in ("sex", "stage", "lauren") if k in COL]
cols = ["Overall"] + order
sub  = {"Overall": df, **{g: df[df["group"] == g] for g in order}}

rows = []
rows.append(["n"] + [str(len(sub[c])) for c in cols])

for k in cont:
    s = {c: pd.to_numeric(sub[c][COL[k]], errors="coerce") for c in cols}
    row = [f"{k}, median (IQR)"]
    for c in cols:
        q1, m, q3 = s[c].quantile([.25, .5, .75])
        row.append(f"{m:.1f} ({q1:.1f}-{q3:.1f})")
    rows.append(row)
    rows.append([f"  {k} missing, n"] + [str(s[c].isna().sum()) for c in cols])

for k in cat:
    levels = df[COL[k]].dropna().unique()
    for lv in sorted(levels, key=str):
        row = [f"{k}: {lv}, n (%)"]
        for c in cols:
            n = (sub[c][COL[k]] == lv).sum()
            row.append(f"{n} ({100 * n / len(sub[c]):.1f}%)")
        rows.append(row)
    rows.append([f"  {k} missing, n"] + [str(sub[c][COL[k]].isna().sum()) for c in cols])

table1 = pd.DataFrame(rows, columns=["Characteristic"] + cols)
table1

,Characteristic,Overall,Co-mutated,TP53 only,Cadherin only,Wild-type
0,n,401,76,107,88,130
1,"age, median (IQR)",67.5 (58.0-73.0),66.0 (58.5-72.0),69.0 (59.2-74.0),69.5 (60.0-73.2),65.0 (57.0-71.5)
2,"age missing, n",5,1,1,0,3
3,"sex: female, n (%)",142 (35.4%),20 (26.3%),41 (38.3%),38 (43.2%),43 (33.1%)
4,"sex: male, n (%)",259 (64.6%),56 (73.7%),66 (61.7%),50 (56.8%),87 (66.9%)
5,"sex missing, n",0,0,0,0,0
6,"stage: I, n (%)",50 (12.5%),12 (15.8%),12 (11.2%),11 (12.5%),15 (11.5%)
7,"stage: II, n (%)",101 (25.2%),18 (23.7%),30 (28.0%),24 (27.3%),29 (22.3%)
8,"stage: III, n (%)",142 (35.4%),27 (35.5%),35 (32.7%),32 (36.4%),48 (36.9%)
9,"stage: IV, n (%)",31 (7.7%),3 (3.9%),9 (8.4%),7 (8.0%),12 (9.2%)


In [36]:
for k in cont:
    gs = [pd.to_numeric(df[df.group == g][COL[k]], errors="coerce").dropna() for g in order]
    print(k, "Kruskal-Wallis p =", round(stats.kruskal(*gs).pvalue, 4))
for k in cat:
    ct = pd.crosstab(df[COL[k]], df["group"])
    print(k, "chi-square p =", round(stats.chi2_contingency(ct)[1], 4))

table1.to_csv("table1_cohort_demographics.csv", index=False)

age Kruskal-Wallis p = 0.0696
sex chi-square p = 0.123
stage chi-square p = 0.9133
